In [0]:
%run ./utils

In [0]:
def transfer_plain_text_file(src, dst):
    dbutils.fs.cp(src,dst)

In [0]:
import os

def _list_s3_recursive(root_path: str):
    """
    Recursively list all file paths under an s3:// prefix using dbutils.fs.ls.
    Returns a flat list of full S3 paths (no directories).
    """
    root = root_path.rstrip('/')
    stack = [root]
    files = []

    while stack:
        current = stack.pop()
        for fi in dbutils.fs.ls(current):
            if fi.isDir():
                stack.append(fi.path.rstrip('/'))
            else:
                files.append(fi.path)
    return files


def replicate_s3_folder_to_volume(
    s3_root: str,
    volume_base: str = "/Volumes/datascience_ea_dev/pe/outputs_for_s3",
):
    """
    Mirror an S3 folder (prefix) into a Databricks Volume, preserving
    folder structure and copying data.

    - If a directory contains parquet parts (e.g. part-*.parquet),
      that whole directory is treated as ONE parquet dataset.
    - CSV/TXT are copied as text.
    - Everything else is copied as binary.
    """
    s3_root = s3_root.rstrip("/")

    # --- 1. Derive the key path after the bucket to replicate that tree ---
    # "s3://bucket-1/folder/folder2" -> bucket="bucket-1", key_prefix="folder/folder2"
    no_scheme = s3_root.replace("s3://", "")
    parts = no_scheme.split("/", 1)
    bucket = parts[0]
    key_prefix = parts[1] if len(parts) > 1 else ""  # may be ""

    volume_root = volume_base.rstrip("/")
    if key_prefix:
        volume_root = os.path.join(volume_root, key_prefix)

    print("Replicating S3 -> Volume")
    print(f"  S3 root     : {s3_root}")
    print(f"  Volume root : {volume_root}")

    dbutils.fs.rm(f"dbfs:{volume_root}", recurse=True)
    dbutils.fs.mkdirs(f"dbfs:{volume_root}")

    # --- 2. List all files under the S3 root ---
    s3_files = _list_s3_recursive(s3_root)
    print(f"Found {len(s3_files)} files under {s3_root}")

    # Normalize prefix for relative path computation
    rel_prefix = s3_root + "/"

    # --- 3. Detect parquet dataset directories (those with part-*.parquet or *.parquet) ---
    parquet_dirs = set()
    for src in s3_files:
        base = os.path.basename(src)
        if base.endswith(".parquet") or (base.startswith("part-") and base.endswith(".parquet")):
            parquet_dirs.add(os.path.dirname(src.rstrip("/")))

    processed_parquet_dirs = set()

    for src in s3_files:
        src = src.rstrip("/")
        parent_dir = os.path.dirname(src)
        base = os.path.basename(src)
        ext = os.path.splitext(base)[1].lower()

        # --- 3.a Handle parquet datasets by directory, once ---
        if parent_dir in parquet_dirs:
            # Only process each parquet directory once
            if parent_dir in processed_parquet_dirs:
                continue

            # directory relative to root
            rel_dir = (
                parent_dir[len(rel_prefix):]
                if parent_dir.startswith(rel_prefix)
                else parent_dir.split("/", 1)[-1]
            )
            dest_dir_local = os.path.join(volume_root, rel_dir)
            dest_dir_dbfs = f"dbfs:{dest_dir_local}"

            dbutils.fs.mkdirs(dest_dir_dbfs)
            print(f"Copying parquet dataset: {parent_dir} -> {dest_dir_local}")

            df = spark.read.parquet(parent_dir)
            df.write.mode("overwrite").parquet(dest_dir_dbfs)

            processed_parquet_dirs.add(parent_dir)
            continue  # skip individual files in this directory (part-*, _SUCCESS, etc.)

        # --- 4. Non-parquet datasets: regular files ---
        # relative path for a single file
        rel_path = (
            src[len(rel_prefix):]
            if src.startswith(rel_prefix)
            else src.split("/", 1)[-1]
        )

        dest_file_local = os.path.join(volume_root, rel_path)
        dest_dir_local = os.path.dirname(dest_file_local)
        dest_dir_dbfs = f"dbfs:{dest_dir_local}"

        dbutils.fs.mkdirs(dest_dir_dbfs)
        print(f"Copying: {src} -> {dest_file_local}")

        # CSV / text
        if ext in [".csv", ".txt"]:
            dbutils.fs.cp(src,dest_file_local)
            # lines = spark.read.text(src).select("value").collect()
            # content = "\n".join(r.value for r in lines)
            # with open(dest_file_local, "w") as f:
            #     f.write(content)

        # (standalone) parquet file, not a dataset directory
        elif ext == ".parquet":
            # Rare case: a solitary parquet file, treat as dataset with one part
            df = spark.read.parquet(src)
            df.write.mode("overwrite").parquet(f"dbfs:{dest_file_local}")

        # Binary / other
        else:
            bdf = spark.read.format("binaryFile").load(src)
            row = bdf.select("content").head()
            if row is None or row[0] is None:
                print(f"  - Skipping empty/unsupported file: {src}")
                continue
            raw = row[0]
            with open(dest_file_local, "wb") as f:
                f.write(bytes(raw))

    print("Done replicating S3 tree to Volume.")


In [0]:
transfer_plain_text_file(
    "s3://memberanalytics-data-out-prod/users/tredence/aryan/propensity_model_for_trips/input_files/features_spend_all_numeric_features.csv",
    feature_path
)

transfer_plain_text_file(
    "s3://memberanalytics-data-out-prod/MODELDATA/BBM_PROPENSITY/BBM_COUPON/coups.csv",
    coups_lookup_path
)

transfer_plain_text_file(
    "s3://memberanalytics-data-out-prod/MODELDATA/BBM_PROPENSITY/BBM_FEATURES/BBM_Propensity_Model_Columns.csv",
    features_lookup_path
)

transfer_plain_text_file(
    "s3://memberanalytics-data-out-prod/USERS/cling/MODELDATA/BBM_PROPENSITY_MODEL/lookup/bbms_history_dates.csv",
    bbm_history_lookup_path
)

In [0]:
if environment.lower() != 'prod':

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/campaign.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/campaign.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/cells.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/cells.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/constructs.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/constructs.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/handshakes.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/handshakes.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/msmt_cells.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/msmt_cells.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/offers.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/offers.csv"
    )

    transfer_plain_text_file(
        "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/segments.csv",
        f"/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/segments.csv"
    )
    df = spark.read.parquet('s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/assignments/')
    df.write.mode('overwrite').format('parquet').partitionBy('cell_id').save(f'/Volumes/{catalog_name}/{pe_schema_name}/outputs_for_s3/ASSIGNMENTS/cdsa/assignments/')

In [0]:
replicate_list = [
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/campaigns/FY27/MMPC05FY27",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/CONSTRUCTS",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/SEGMENTS",
    "s3://memberanalytics-data-out-prod/AD_HOC/same_day_delivery_zip_codes",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/assn_output/PROD/FY21/BBM11FY21/final_2020-06-15",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_quals",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_map",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_bank",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_map_deleted",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_quals_deleted",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/cdsa/coupons/PROD/coupon_bank_deleted",
    "s3://memberanalytics-data-out-prod/ASSIGNMENTS/logs/create_coupons",
]

if environment.lower() != 'prod':
    for path in replicate_list:
        replicate_s3_folder_to_volume(
            s3_root=path,
            volume_base=f"/Volumes/{catalog_name}/pe/outputs_for_s3"
        )

